In [0]:
%run ../read_params

In [0]:
%run ../utils

In [0]:
def get_pokemon_locations(game_version_name):

    pokedex_version = spark.sql(f"""
        SELECT 
            ep.region
            ,gv.version_name
            ,np.pokedex_number AS national_pokedex_number
            ,ep.pokemon_pokedex_number
            ,ep.pokemon_name
            ,s.generation
            ,ep.pokemon_url
        FROM 
            {BRONZE_DATABASE_PREFIX}.game_versions gv

        LEFT JOIN {BRONZE_DATABASE_PREFIX}.expanded_pokedexes ep
        ON gv.version_group_id = ep.id

        LEFT JOIN {BRONZE_DATABASE_PREFIX}.national_pokedex np
        ON ep.pokemon_url = np.species_url

        LEFT JOIN {BRONZE_DATABASE_PREFIX}.species s
        ON np.pokedex_number = s.nat_dex_pokedex_no

        WHERE 
            gv.version_name = '{game_version_name}'
    """)

    region, version_name = pokedex_version.select('region', 'version_name').distinct().collect()[0]

    table_name = f"pokedex__{region}__{version_name}"

    pokedex_version.write.format('delta').mode("overwrite").option('overwriteSchema', 'true').saveAsTable(f"{SILVER_DATABASE_PREFIX}.{table_name}")

    encounters = spark.sql(f"""
        SELECT 
            nat_dex_pokedex_no
            ,species_id
            ,variety_id
            ,encounter_location_area
            ,encounter_rate
            ,encounter_method
            ,encounter_condition_values
            ,encounter_min_level
            ,encounter_max_level
        FROM 
            {BRONZE_DATABASE_PREFIX}.encounters                   
        WHERE 
            REPLACE(version_name, '-', '_') = '{version_name}'
    """)

    pokedex_locations = pokedex_version.alias('pd').join(
        encounters.alias('e'), 
        col('pd.national_pokedex_number') == col('e.nat_dex_pokedex_no'), 
        'left'
    )

    locations_table_name = f"pokemon_locations__{region}__{version_name}"

    pokedex_locations.write.format('delta').mode("overwrite").option('overwriteSchema', 'true').saveAsTable(f"{SILVER_DATABASE_PREFIX}.{locations_table_name}")

In [0]:
game_versions_list = [row['version_name'] for row in spark.table(f"bronze.game_versions").select("version_name").distinct().collect()]

for game_version_name in game_versions_list:
    print(f'Creating the pokedex and encounter tables for {game_version_name}...')
    get_pokemon_locations(game_version_name)